In [44]:
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [45]:

# Set the path to the file you'd like to load
# Para este dataset específico, el archivo se llama 'credit_risk_dataset.csv'
file_path = "loan/loan.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "ranadeep/credit-risk-dataset",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())


/tmp/ipykernel_31542/358505134.py:6: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(
/home/tomasgomez/Github/RedesNeuronales_Trabajo2/.venv/lib64/python3.14/site-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (19: desc, 55: verification_status_joint) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


First 5 records:         id  member_id  loan_amnt  funded_amnt  funded_amnt_inv        term  \
0  1077501    1296599     5000.0       5000.0           4975.0   36 months   
1  1077430    1314167     2500.0       2500.0           2500.0   60 months   
2  1077175    1313524     2400.0       2400.0           2400.0   36 months   
3  1076863    1277178    10000.0      10000.0          10000.0   36 months   
4  1075358    1311748     3000.0       3000.0           3000.0   60 months   

   int_rate  installment grade sub_grade  ... total_bal_il il_util  \
0     10.65       162.87     B        B2  ...          NaN     NaN   
1     15.27        59.83     C        C4  ...          NaN     NaN   
2     15.96        84.33     C        C5  ...          NaN     NaN   
3     13.49       339.31     C        C1  ...          NaN     NaN   
4     12.69        67.79     B        B5  ...          NaN     NaN   

  open_rv_12m  open_rv_24m max_bal_bc all_util total_rev_hi_lim inq_fi  \
0         NaN      

In [46]:
# 1. Definir el diccionario de mapeo para 0 y 1
mapping = {
    'Fully Paid': 0,
    'Does not meet the credit policy. Status:Fully Paid': 0,
    'Charged Off': 1,
    'Default': 1,
    'Late (31-120 days)': 1,
    'Does not meet the credit policy. Status:Charged Off': 1
}

# 2. Filtrar únicamente los casos válidos (excluyendo los NA)
df = df[df['loan_status'].isin(mapping.keys())].copy()
print(df['loan_status'].head())
# 3. Crear la nueva variable objetivo binaria
df['loan_status'] = df['loan_status'].map(mapping)
print(df['loan_status'].head())

0     Fully Paid
1    Charged Off
2     Fully Paid
3     Fully Paid
5     Fully Paid
Name: loan_status, dtype: str
0    0
1    1
2    0
3    0
5    0
Name: loan_status, dtype: int64


In [48]:
columnas_a_excluir = ["url", "last_pymnt_d", "next_pymnt_d", "emp_title", "issue_d", "desc", "title", "zip_code", "addr_state", "earliest_cr_line", "last_credit_pull_d"]
df = df.drop(columns=columnas_a_excluir)

In [65]:
# Muestra el número de valores únicos por cada columna de texto o categórica
df.select_dtypes(include=['object', 'category']).nunique()

/tmp/ipykernel_31542/2292819751.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include=['object', 'category']).nunique()


term                    2
home_ownership          6
pymnt_plan              2
purpose                14
initial_list_status     2
application_type        2
dtype: int64

In [63]:
print("emp_length: " + str(list((df['emp_length'].unique()))))
print("grade: " + str(list((df['grade'].unique()))))
print("sub_grade: " + str(list((df['sub_grade'].unique()))))
print("verification_status: " + str(list((df['verification_status'].unique()))))
print("verification_status_joint: " + str(list((df['verification_status_joint'].unique()))))

emp_length: ['10+ years', '< 1 year', '3 years', '9 years', '4 years', '5 years', '1 year', '6 years', '2 years', '7 years', '8 years', nan]
grade: [np.int64(1), np.int64(2), np.int64(0), np.int64(4), np.int64(5), np.int64(3), np.int64(6)]
sub_grade: [np.int64(6), np.int64(13), np.int64(14), np.int64(10), np.int64(3), np.int64(20), np.int64(26), np.int64(9), np.int64(12), np.int64(5), np.int64(15), np.int64(0), np.int64(7), np.int64(8), np.int64(11), np.int64(16), np.int64(2), np.int64(4), np.int64(19), np.int64(1), np.int64(23), np.int64(17), np.int64(18), np.int64(27), np.int64(22), np.int64(25), np.int64(24), np.int64(33), np.int64(21), np.int64(31), np.int64(30), np.int64(29), np.int64(28), np.int64(34), np.int64(32)]
verification_status: [np.int64(2), np.int64(1), np.int64(0)]
verification_status_joint: [np.float64(nan), np.float64(1.0), np.float64(0.0), np.float64(2.0)]


In [64]:
# 1. Mapeo para 'grade' (de menor a mayor riesgo, A es mejor que G)
grade_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}
df['grade'] = df['grade'].map(grade_mapping)

# 2. Mapeo automático/ordenado para 'sub_grade' (A1 a G5)
letras = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
subgrade_mapping = {f"{l}{n}": i for i, (l, n) in enumerate(
    (l, n) for l in letras for n in range(1, 6)
)}
df['sub_grade'] = df['sub_grade'].map(subgrade_mapping)

# 3. Mapeo para 'verification_status' (ordenado por nivel de rigor)
verification_mapping = {'Not Verified': 0, 'Source Verified': 1, 'Verified': 2}
df['verification_status'] = df['verification_status'].map(verification_mapping)

# 4. Mapeo para 'verification_status_joint' (manejando los nulos con -1 o NaN)
verification_joint_mapping = {
    'Not Verified': 0, 'Source Verified': 1, 'Verified': 2
}
df['verification_status_joint'] = df['verification_status_joint'].map(
    verification_joint_mapping
)

# Mapeo ordenado para 'emp_length' (los nulos se pueden dejar como -1 o NaN)
emp_length_mapping = {
    '< 1 year': 0,
    '1 year': 1,
    '2 years': 2,
    '3 years': 3,
    '4 years': 4,
    '5 years': 5,
    '6 years': 6,
    '7 years': 7,
    '8 years': 8,
    '9 years': 9,
    '10+ years': 10
}

# Aplicar el mapeo conservando el nombre de la columna
df['emp_length'] = df['emp_length'].map(emp_length_mapping)

In [68]:
# Selección de columnas no numéricas:
columns_to_dummy = df.select_dtypes(exclude=['number'])

# Vista de columnas no numéricas.
columns_to_dummy.head()

,term,home_ownership,pymnt_plan,purpose,initial_list_status,application_type
0,36 months,RENT,n,credit_card,f,INDIVIDUAL
1,60 months,RENT,n,car,f,INDIVIDUAL
2,36 months,RENT,n,small_business,f,INDIVIDUAL
3,36 months,RENT,n,other,f,INDIVIDUAL
5,36 months,RENT,n,wedding,f,INDIVIDUAL


In [69]:
# Función auxiliar (generada con Gemini)
# Codificación con one-hot pero utilizando -1/1 en lugar de 0/1.
def dummy_encode(df, column):
    """
    Encodes a categorical column into dummy variables using -1/1 encoding.

    Args:
        df: The input DataFrame.
        column: The name of the column to encode.

    Returns:
        A DataFrame with the dummy variables.
    """

    dummies = pd.get_dummies(df[column], prefix=column, prefix_sep='_', drop_first=True)
    dummies = dummies.map(lambda x: 1 if x == 1 else -1)  # Convert 0s to -1s
    return pd.concat([df.drop(column, axis=1), dummies], axis=1)

In [70]:
# Aplicación de la función auxiliar para hacer el one-hot encoding:
dummy_encoded = columns_to_dummy.copy()

for column in dummy_encoded.columns:
    dummy_encoded = dummy_encode(dummy_encoded, column)

# Visualización del dataframe recodificado
dummy_encoded.head()

,term_ 60 months,home_ownership_MORTGAGE,home_ownership_NONE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,pymnt_plan_y,purpose_credit_card,purpose_debt_consolidation,purpose_educational,...,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding,initial_list_status_w,application_type_JOINT
0,-1,-1,-1,-1,-1,1,-1,1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
1,1,-1,-1,-1,-1,1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
2,-1,-1,-1,-1,-1,1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,1,-1,-1,-1,-1
3,-1,-1,-1,-1,-1,1,-1,-1,-1,-1,...,-1,-1,-1,1,-1,-1,-1,-1,-1,-1
5,-1,-1,-1,-1,-1,1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,1,-1,-1


In [ ]:
# Selección de las columnas numéricas:
numeric_cols = datos.drop(columns_to_dummy.columns.tolist(), axis=1)

# Display the first few rows of the selected numeric columns
print(numeric_cols.head())
